# Stage B / NB 08 — BiomedCLIP diagnostic-entity agent (A6)

Protocol reference: agent **A6** (reinstated, see the note below); experiments **E0i_entity**,
**E1-L6**. Answers referee 1.2 directly, including the specific ablation the referee asked for:

> *"MobileViT + DinoV2 only vs. MobileViT + DinoV2 + BiomedCLIP"*
> *"the contribution of each component (e.g. YOLOv8 cropping, BiomedCLIP embeddings, ...) is
> not fully disentangled"*

## Why BiomedCLIP is back in the roster

The resubmission plan originally folded BiomedCLIP's role into CXformer (agent A5) and dropped
it. That was a mistake for two reasons.

1. **Referee 1.2 names it.** Deleting a component the referee asked us to ablate means we can
   only say "we replaced it", not "here is what it contributed". `E1-L6` restores the ability
   to answer the question that was actually asked.
2. **It supplies a capability no other agent has.** CXformer returns a 768-d vector; the
   LoRA VLMs return scores. BiomedCLIP is a vision–**language** contrastive model, so it returns
   *named findings with confidences* — "bilateral ground-glass opacity, 0.82". For a framework
   whose selling point is auditable reasoning, structured named evidence is materially more
   useful as reasoner input than an embedding, and it is what makes the E3 prompt readable.

This does **not** reinstate `google/cxr-foundation` (still withdrawn, D2a): that was redundant
with the MedGemma vision tower, whereas BiomedCLIP is architecturally and functionally distinct.

## Numbering

The pre-existing `notebook_implementation_plan.txt` numbered this notebook 08, and this file
keeps that number. The resubmission plan's Section 9 therefore shifts by one from here on:
MedGemma LoRA → NB 09, Qwen LoRA → NB 10, NV-Reason → NB 11, anatomy-aware → NB 12, and so on.

## What the agent produces

| output | consumer |
| --- | --- |
| per-image entity scores over a fixed vocabulary | NB 13 registry, NB 15 reasoner prompt |
| thresholded structured findings JSON | E3b/E3c evidence channel, E10c grounding audit |
| entity-feature probe predictions (COVID + mRALE) | Table 2 row, E1-L6 leave-one-out |
| probe coefficients per entity | **E10 interpretability**: which named findings drive severity |

That last row is worth noting. A linear probe over ~30 named findings is inherently
interpretable: its coefficients say "extensive bilateral opacity pushes mRALE up, clear lungs
pushes it down". That is a genuine, cheap contribution to referee 1.4 (interpretability) that no
other agent in the roster provides.

## Outputs (under `stage_B/nb08_biomedclip_entity/`)
- `entity_scores.parquet`, `entity_findings.jsonl` (structured, for the reasoner)
- `predictions_entity_probe.jsonl`, `arm_summary.csv`, `probe_coefficients.csv`
- `construct_validity.csv`, `entity_vocabulary.json`
- `external_predictions.jsonl`, `run_config.json`, `gate_nb08.json`

## Gate
- Model loads, all entity prompts encode.
- Entity coverage is 100% of internal images.
- **Construct validity**: opacity entities correlate positively with mRALE and "clear lungs"
  correlates negatively, both beyond a pre-registered floor. With no entity ground truth
  available, this is the check that distinguishes a working agent from a noise generator.

## 1. Imports, seeds, and the Stage A path contract

In [ ]:
import gc
import json
import math
import os
import random
import sys
import time
from collections import Counter, OrderedDict, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# Shared metric definitions. Table 2 is only a valid comparison if every arm uses these.
_METRICS_SEARCH = [Path.cwd(), Path.cwd().parent, Path.cwd() / "stage_B",
                   Path.cwd().parent / "stage_B"]
for _candidate in _METRICS_SEARCH:
    if (_candidate / "cxr_metrics.py").is_file():
        sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError(
        "cxr_metrics.py not found. It must sit beside the Stage B notebooks; every arm in "
        f"Table 2 depends on its metric definitions. Searched: {_METRICS_SEARCH}")
import cxr_metrics as cm

SEED = 42
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# ---- Stage A path contract -------------------------------------------------------------
FALLBACK_STAGE_A_DIR = Path("/data/liangz2/openi/midrc/tetci_resubmit/stage_A")
PATHS_JSON_CANDIDATES = [
    FALLBACK_STAGE_A_DIR / "nb00_environment" / "stage_a_paths.json",
    Path.cwd() / "stage_a_paths.json",
    Path.cwd().parent / "stage_A" / "nb00_environment" / "stage_a_paths.json",
]
stage_paths = None
for candidate in PATHS_JSON_CANDIDATES:
    if candidate.is_file():
        stage_paths = json.loads(candidate.read_text(encoding="utf-8"))
        print("Path contract:", candidate)
        break
if stage_paths is None:
    raise FileNotFoundError("stage_a_paths.json not found. Run Stage A NB 00 first.")

PROJECT_ROOT = Path(stage_paths["project_root"])
STAGE_ROOT = Path(stage_paths["stage_root"])
STAGE_A_DIR = Path(stage_paths["stage_a_dir"])
STAGE_B_DIR = STAGE_ROOT / "stage_B"
NB01_DIR = Path(stage_paths["nb_output_dirs"]["nb01_inventory"])
NB02_DIR = Path(stage_paths["nb_output_dirs"]["nb02_folds"])
NB03_DIR = Path(stage_paths["nb_output_dirs"]["nb03_external"])
NB04_DIR = Path(stage_paths["nb_output_dirs"]["nb04_localization"])
FOLD_DEF_DIR = NB02_DIR / "fold_definitions"
MODEL_REVISIONS = stage_paths.get("model_revisions", {})

N_FOLDS = 5
INTERNAL_VALIDATION_FRACTION = 0.10   # matches the tested LoRA notebooks

print("Stage B output root:", STAGE_B_DIR)
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| BF16:", torch.cuda.is_bf16_supported())

## 2. Configuration

In [ ]:
import hashlib

NB08_DIR = STAGE_B_DIR / "nb08_biomedclip_entity"
NB08_DIR.mkdir(parents=True, exist_ok=True)

BIOMEDCLIP_ID = "microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
BIOMEDCLIP_FALLBACK_REVISION = "9f341de24bfb00180f1b847274256e9b65a3a32e"
BIOMEDCLIP_REVISION = (MODEL_REVISIONS.get(BIOMEDCLIP_ID)
                       or BIOMEDCLIP_FALLBACK_REVISION)
BIOMEDCLIP_REVISION_SOURCE = ("stage_a_paths.json"
                               if MODEL_REVISIONS.get(BIOMEDCLIP_ID)
                               else "NB08 pinned fallback")

AGENT_NAME = "A6_biomedclip"
ENTITY_ARM = "E0i_entity_probe"

IMAGE_BATCH_SIZE = 32
# Entity scores are flushed every this-many images (atomic .tmp rename), so an
# interruption costs at most this much encoding rather than the whole cohort.
SCORE_FLUSH_EVERY = 512
SCORE_VIEW = "v0_image"          # whole image; regional entity scoring is a later extension

# Thresholds for turning continuous scores into the structured findings the reasoner reads.
FINDING_THRESHOLD = 0.50
FINDING_MAX_REPORTED = 8         # keep the reasoner prompt short and auditable

# Probe configuration. Deliberately linear: the coefficients ARE the interpretability output.
PROBE_C = 1.0                    # inverse regularisation for logistic regression
PROBE_ALPHA = 1.0                # ridge penalty for the ordinal/continuous components
STANDARDISE_FEATURES = True

# Construct-validity floors, pre-registered here so the gate is not tuned after seeing results.
MIN_OPACITY_MRALE_SPEARMAN = 0.20     # opacity entities vs mRALE total
MAX_CLEAR_LUNG_MRALE_SPEARMAN = -0.15 # "clear lungs" must anti-correlate

RUN_EXTERNAL = True
MAX_IMAGES = None                # set 64 for a wiring check, then None

print("Model:", BIOMEDCLIP_ID)
print("Revision:", BIOMEDCLIP_REVISION, f"({BIOMEDCLIP_REVISION_SOURCE})")
print("Output:", NB08_DIR)

## 3. Entity vocabulary

Fixed before any scoring, and recorded in `entity_vocabulary.json`. Five groups, each with a
stated purpose. Every entity is scored as a **contrastive pair** — a positive phrase against an
explicit negative phrase — rather than by an absolute similarity threshold, because raw CLIP
cosine similarities are not calibrated across prompts and cannot be compared to a fixed cutoff.

The `support_device` group is not decoration. Portable ICU films carry lines and tubes, and
sicker patients get more of them, so device presence is a **severity proxy that a model can
exploit without reading the lungs at all**. Scoring devices explicitly turns that confound from
an invisible shortcut into a measurable quantity — and if device scores predict mRALE better
than opacity scores do, that is something the paper needs to disclose, not discover in review.

In [ ]:
# group -> {entity: (positive phrase, negative phrase)}
ENTITY_VOCABULARY = OrderedDict([
    ("parenchymal_opacity", OrderedDict([
        ("consolidation", ("chest x-ray with airspace consolidation",
                           "chest x-ray with no consolidation")),
        ("ground_glass_opacity", ("chest x-ray with ground glass opacity",
                                  "chest x-ray with no ground glass opacity")),
        ("hazy_opacity", ("chest x-ray with hazy pulmonary opacity",
                          "chest x-ray with clear lung fields")),
        ("reticular_opacity", ("chest x-ray with reticular interstitial opacity",
                               "chest x-ray with no interstitial opacity")),
        ("dense_opacity", ("chest x-ray with dense white-out opacification",
                           "chest x-ray with normally aerated lungs")),
    ])),
    ("extent_distribution", OrderedDict([
        ("bilateral_involvement", ("chest x-ray with bilateral lung opacities",
                                   "chest x-ray with unilateral or no lung opacity")),
        ("peripheral_distribution", ("chest x-ray with peripheral subpleural opacities",
                                     "chest x-ray with central perihilar opacities")),
        ("lower_zone_predominant", ("chest x-ray with lower zone predominant opacity",
                                    "chest x-ray with upper zone predominant opacity")),
        ("extensive_involvement", ("chest x-ray with extensive opacity involving most of both lungs",
                                   "chest x-ray with minimal or focal opacity")),
    ])),
    ("other_pathology", OrderedDict([
        ("pleural_effusion", ("chest x-ray with pleural effusion",
                              "chest x-ray with no pleural effusion")),
        ("pneumothorax", ("chest x-ray with pneumothorax",
                          "chest x-ray with no pneumothorax")),
        ("cardiomegaly", ("chest x-ray with enlarged cardiac silhouette",
                          "chest x-ray with normal heart size")),
        ("atelectasis", ("chest x-ray with atelectasis", "chest x-ray with no atelectasis")),
        ("nodule_or_mass", ("chest x-ray with a pulmonary nodule or mass",
                            "chest x-ray with no pulmonary nodule or mass")),
        ("cavitation", ("chest x-ray with a cavitary lung lesion",
                        "chest x-ray with no cavitation")),
        ("upper_lobe_infiltrate", ("chest x-ray with upper lobe infiltrate suggesting tuberculosis",
                                   "chest x-ray with no upper lobe infiltrate")),
    ])),
    ("normality", OrderedDict([
        ("clear_lungs", ("normal chest x-ray with clear lungs",
                         "abnormal chest x-ray with lung opacity")),
        ("normal_study", ("normal chest radiograph with no acute cardiopulmonary abnormality",
                          "abnormal chest radiograph with acute findings")),
    ])),
    # Severity proxies that do not require reading the lungs. Measured on purpose.
    ("support_device", OrderedDict([
        ("endotracheal_tube", ("chest x-ray with an endotracheal tube",
                               "chest x-ray with no endotracheal tube")),
        ("central_line", ("chest x-ray with a central venous catheter",
                          "chest x-ray with no central venous catheter")),
        ("nasogastric_tube", ("chest x-ray with a nasogastric tube",
                              "chest x-ray with no nasogastric tube")),
        ("ecg_leads", ("portable chest x-ray with ECG leads and monitoring wires",
                       "chest x-ray with no external monitoring leads")),
    ])),
])

# Prompt ensembling: several phrasings per entity, averaged in embedding space. Standard CLIP
# practice; it measurably reduces sensitivity to any single wording, which matters because
# referee 1.1 asked for prompt-design justification.
PROMPT_TEMPLATES = [
    "{}",
    "a frontal chest radiograph showing {}",
    "this chest x-ray demonstrates {}",
    "findings: {}",
]

ENTITY_INDEX = []
for group, entities in ENTITY_VOCABULARY.items():
    for entity in entities:
        ENTITY_INDEX.append((group, entity))
ENTITY_NAMES = [entity for _, entity in ENTITY_INDEX]

OPACITY_ENTITIES = (list(ENTITY_VOCABULARY["parenchymal_opacity"])
                    + list(ENTITY_VOCABULARY["extent_distribution"]))
DEVICE_ENTITIES = list(ENTITY_VOCABULARY["support_device"])
NORMALITY_ENTITIES = list(ENTITY_VOCABULARY["normality"])

print(f"Entities: {len(ENTITY_NAMES)} across {len(ENTITY_VOCABULARY)} groups")
for group, entities in ENTITY_VOCABULARY.items():
    print(f"  {group:<22} {len(entities):>2}  {', '.join(list(entities)[:4])}"
          + (" ..." if len(entities) > 4 else ""))
print(f"Prompt templates per phrase: {len(PROMPT_TEMPLATES)}")
print(f"Total text encodings: {len(ENTITY_NAMES) * 2 * len(PROMPT_TEMPLATES)}")

cm.write_json(NB08_DIR / "entity_vocabulary.json", {
    "vocabulary": {group: {entity: {"positive": pair[0], "negative": pair[1]}
                           for entity, pair in entities.items()}
                   for group, entities in ENTITY_VOCABULARY.items()},
    "prompt_templates": PROMPT_TEMPLATES,
    "scoring": "contrastive pair softmax over (positive, negative) prompt ensembles",
    "groups": {
        "parenchymal_opacity": "mRALE density axis",
        "extent_distribution": "mRALE extent axis",
        "other_pathology": "differential findings; drives the Montgomery TB contrast",
        "normality": "negative control; must anti-correlate with mRALE",
        "support_device": "severity proxy / confound, measured explicitly",
    },
})

## 4. Load BiomedCLIP

In [ ]:
from PIL import Image

Image.MAX_IMAGE_PIXELS = None
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_biomedclip():
    errors = {}
    # open_clip is the reference implementation for BiomedCLIP (PubMedBERT text tower +
    # ViT-B/16 image tower), so it is tried first.
    try:
        from huggingface_hub import snapshot_download
        from open_clip import create_model_from_pretrained, get_tokenizer
        snapshot = snapshot_download(repo_id=BIOMEDCLIP_ID, revision=BIOMEDCLIP_REVISION)
        local_model = f"local-dir:{snapshot}"
        model, preprocess = create_model_from_pretrained(local_model)
        tokenizer = get_tokenizer(local_model)
        model = model.to(device).eval()
        return model, preprocess, tokenizer, "open_clip"
    except Exception as exc:
        errors["open_clip"] = f"{type(exc).__name__}: {exc}"

    try:
        from transformers import AutoModel, AutoProcessor
        kwargs = {"trust_remote_code": True}
        if BIOMEDCLIP_REVISION:
            kwargs["revision"] = BIOMEDCLIP_REVISION
        processor = AutoProcessor.from_pretrained(BIOMEDCLIP_ID, **kwargs)
        model = AutoModel.from_pretrained(BIOMEDCLIP_ID, **kwargs).to(device).eval()
        return model, processor, processor, "transformers"
    except Exception as exc:
        errors["transformers"] = f"{type(exc).__name__}: {exc}"

    raise RuntimeError(
        "BiomedCLIP could not be loaded. Agent A6 and the E1-L6 ablation both depend on it, "
        "and E1-L6 is what answers referee 1.2's named request. Install open_clip_torch "
        "(pip install open_clip_torch) rather than substituting another model.\n"
        f"{json.dumps(errors, indent=2)}"
    )


model, preprocess, tokenizer, backend = load_biomedclip()
n_parameters = sum(p.numel() for p in model.parameters())
print(f"Loaded BiomedCLIP via {backend}: {n_parameters / 1e6:.1f}M parameters (frozen)")
print("Pinned snapshot loaded:", BIOMEDCLIP_REVISION)

## 5. Encode the entity prompts

Positive and negative phrases are each averaged over the template ensemble, then L2-normalised.
Encoding happens once for the whole study — text embeddings do not depend on the image.

In [ ]:
@torch.inference_mode()
def encode_texts(phrases):
    if backend == "open_clip":
        tokens = tokenizer(phrases, context_length=256).to(device)
        features = model.encode_text(tokens)
    else:
        batch = tokenizer(text=phrases, return_tensors="pt", padding=True,
                          truncation=True, max_length=256).to(device)
        features = model.get_text_features(**batch)
    return torch.nn.functional.normalize(features.float(), dim=-1)


positive_embeddings, negative_embeddings = [], []
for group, entity in ENTITY_INDEX:
    positive_phrase, negative_phrase = ENTITY_VOCABULARY[group][entity]
    positive = encode_texts([template.format(positive_phrase)
                             for template in PROMPT_TEMPLATES]).mean(dim=0)
    negative = encode_texts([template.format(negative_phrase)
                             for template in PROMPT_TEMPLATES]).mean(dim=0)
    positive_embeddings.append(torch.nn.functional.normalize(positive, dim=-1))
    negative_embeddings.append(torch.nn.functional.normalize(negative, dim=-1))

positive_matrix = torch.stack(positive_embeddings)          # (n_entities, dim)
negative_matrix = torch.stack(negative_embeddings)
print(f"Encoded {positive_matrix.shape[0]} entity pairs, dim={positive_matrix.shape[1]}")

# Sanity check: positive and negative phrases must actually separate in embedding space. If a
# pair's embeddings are nearly identical, the contrast carries no information and the entity's
# score will be noise around 0.5 regardless of the image.
pair_similarity = (positive_matrix * negative_matrix).sum(dim=-1).cpu().numpy()
degenerate = [(ENTITY_NAMES[index], float(value))
              for index, value in enumerate(pair_similarity) if value > 0.98]
print(f"Mean positive-negative cosine similarity: {pair_similarity.mean():.4f} "
      f"(range {pair_similarity.min():.4f}-{pair_similarity.max():.4f})")
if degenerate:
    print("WARNING: near-degenerate prompt pairs (positive and negative almost identical):")
    for name, value in degenerate:
        print(f"    {name}: {value:.4f}  -> rewrite the negative phrase")
else:
    print("All prompt pairs are separable.")

## 6. Score every image

In [ ]:
def load_view_index():
    path = NB04_DIR / "view_index.csv"
    if not path.is_file():
        raise FileNotFoundError(f"{path} not found. Run Stage A NB 04 first.")
    frame = pd.read_csv(path)
    print(f"view_index.csv: {len(frame):,} rows, cohorts={dict(Counter(frame['cohort']))}")
    return frame


def load_folds():
    path = FOLD_DEF_DIR / "midrc_folds_v2.csv"
    if not path.is_file():
        raise FileNotFoundError(
            f"{path} not found. Run Stage A NB 02 first. Do NOT fall back to the legacy "
            "multi_task_CV folds: they leak at study level."
        )
    frame = pd.read_csv(path)
    print(f"midrc_folds_v2.csv: {len(frame):,} images, "
          f"{frame['group_id'].nunique():,} groups, folds={dict(sorted(Counter(frame['fold']).items()))}")
    return frame


def build_cohort_table():
    # One row per image: labels + fold + every cached view path. This is the single table
    # every Stage B notebook trains and predicts from.
    views = load_view_index()
    folds = load_folds()

    internal = folds.merge(
        views[views["cohort"] == "MIDRC"].drop(columns=["held_out_fold"], errors="ignore"),
        on="filename", how="inner", suffixes=("", "_view"),
    )
    if len(internal) != len(folds):
        missing = set(folds["filename"]) - set(internal["filename"])
        raise RuntimeError(
            f"{len(missing)} fold images have no NB 04 localization row (e.g. "
            f"{sorted(missing)[:5]}). Re-run NB 04 with MAX_IMAGES=None."
        )
    internal["mrale_right"] = (internal["extent_right_numerical"]
                               * internal["density_right_numerical"])
    internal["mrale_left"] = (internal["extent_left_numerical"]
                              * internal["density_left_numerical"])
    internal["is_external"] = False

    external_rows = []
    external_dir = NB03_DIR / "external_manifests"
    if external_dir.is_dir():
        for manifest_path in sorted(external_dir.glob("*_manifest.csv")):
            frame = pd.read_csv(manifest_path)
            if "status" in frame.columns:
                frame = frame[frame["status"] == "OK"]
            if not len(frame):
                continue
            cohort = str(frame["cohort"].iloc[0])
            merged = frame.merge(
                views[views["cohort"] == cohort][
                    ["filename", "v0_image", "v1_thorax_image", "v2_left_image",
                     "v2_right_image", "left_box", "right_box", "any_fallback"]
                ],
                on="filename", how="inner",
            )
            merged["fold"] = -1
            merged["is_external"] = True
            merged["group_id"] = "external::" + merged["filename"].astype(str)
            for column in ["mrale_total_annotated", "mrale_right", "mrale_left",
                           "extent_right_numerical", "density_right_numerical",
                           "extent_left_numerical", "density_left_numerical"]:
                if column not in merged.columns:
                    merged[column] = np.nan
            if "mrale_total" in merged.columns:
                merged["mrale_total_annotated"] = merged["mrale_total"]
            external_rows.append(merged)

    table = pd.concat([internal] + external_rows, ignore_index=True, sort=False)
    table["image_key"] = table.apply(
        lambda row: f"{row.get('cohort', 'MIDRC')}::{row['filename']}", axis=1)
    print()
    print(f"Cohort table: {len(table):,} rows "
          f"({int((~table['is_external']).sum()):,} internal, "
          f"{int(table['is_external'].sum()):,} external)")
    return table


def grouped_inner_split(subset, fraction, seed):
    # Group-aware inner validation split, same construction as the tested notebooks: whole
    # groups move together so the inner split cannot leak either.
    groups = sorted(subset["group_id"].astype(str).unique())
    rng = random.Random(seed)
    rng.shuffle(groups)
    n_validation = max(1, round(len(groups) * fraction))
    validation_groups = set(groups[:n_validation])
    is_validation = subset["group_id"].astype(str).isin(validation_groups)
    train, validation = subset[~is_validation], subset[is_validation]
    assert not (set(train["group_id"]) & set(validation["group_id"]))
    return train, validation


def ground_truth_fields(row):
    def maybe_int(value):
        return None if value is None or (isinstance(value, float) and math.isnan(value)) else int(value)
    covid = row.get("covid_positive")
    if isinstance(covid, float) and math.isnan(covid):
        covid = None
    return {
        "gt_covid": covid if covid in {"Yes", "No"} else None,
        "gt_mrale_total": maybe_int(row.get("mrale_total_annotated")),
        "gt_mrale_right": maybe_int(row.get("mrale_right")),
        "gt_mrale_left": maybe_int(row.get("mrale_left")),
        "gt_extent_right": maybe_int(row.get("extent_right_numerical")),
        "gt_density_right": maybe_int(row.get("density_right_numerical")),
        "gt_extent_left": maybe_int(row.get("extent_left_numerical")),
        "gt_density_left": maybe_int(row.get("density_left_numerical")),
    }


def evaluate_arm(rows, label):
    # Single entry point for metrics, so every arm in Table 2 is scored identically.
    covid_rows = [row for row in rows if row.get("gt_covid") is not None]
    metrics = {"arm": label, "n_rows": len(rows)}
    if covid_rows:
        metrics["covid"] = cm.classification_metrics(
            [row["gt_covid"] for row in covid_rows],
            [row.get("covid_pred") for row in covid_rows],
            [row.get("covid_score") for row in covid_rows],
        )
    mrale_rows = [row for row in rows if row.get("gt_mrale_total") is not None]
    if mrale_rows:
        metrics["mrale"] = cm.mrale_metrics(mrale_rows)
    metrics["output"] = cm.localization_free_metrics(rows)
    return metrics


def print_arm_summary(metrics):
    covid = metrics.get("covid", {})
    mrale = metrics.get("mrale", {})
    print(f"  {metrics['arm']:<34} "
          f"AUROC={covid.get('auroc', float('nan')):.4f} "
          f"balAcc={covid.get('balanced_accuracy', float('nan')):.4f} "
          f"spec={covid.get('specificity', float('nan')):.4f} | "
          f"mRALE MAE={mrale.get('mae', float('nan')):.3f} "
          f"QWK={mrale.get('qwk', float('nan')):.4f} "
          f"cov={mrale.get('coverage', float('nan')):.3f}")

In [ ]:
cohort = build_cohort_table()
if not RUN_EXTERNAL:
    cohort = cohort[~cohort["is_external"]]
if MAX_IMAGES is not None:
    internal_sample = cohort[~cohort["is_external"]].groupby("fold", group_keys=False).head(
        max(2, MAX_IMAGES // N_FOLDS))
    cohort = pd.concat([internal_sample, cohort[cohort["is_external"]].head(8)],
                       ignore_index=True)
    print(f"WIRING CHECK: {len(cohort)} images")
cohort = cohort.sort_values("image_key").reset_index(drop=True)
internal_cohort = cohort[~cohort["is_external"]].reset_index(drop=True)
external_cohort = cohort[cohort["is_external"]].reset_index(drop=True)
print(f"Scoring {len(cohort):,} images "
      f"({len(internal_cohort):,} internal, {len(external_cohort):,} external)")

In [ ]:
SCORE_CACHE = NB08_DIR / "entity_scores.npz"

# The cache key must cover EVERYTHING that changes a score, not just the entity names.
# Comparing names alone was a real bug: rewriting a positive/negative PHRASE leaves the names
# untouched, so the stale scores would be reused silently -- and rewriting a negative phrase is
# exactly what the degenerate-pair gate tells you to do, so its own remediation would appear
# not to work. Prompt templates, pooling, view, and model revision matter for the same reason.
SCORE_CACHE_FINGERPRINT = hashlib.sha256(json.dumps({
    "vocabulary": {group: {entity: list(pair) for entity, pair in entities.items()}
                   for group, entities in ENTITY_VOCABULARY.items()},
    "prompt_templates": PROMPT_TEMPLATES,
    "score_view": SCORE_VIEW,
    "model_id": BIOMEDCLIP_ID,
    "model_revision": BIOMEDCLIP_REVISION,
}, sort_keys=True).encode()).hexdigest()[:16]
print(f"Score-cache fingerprint: {SCORE_CACHE_FINGERPRINT}")

# Recover a valid partial checkpoint written by the older implementation.
# np.savez_compressed() appended '.npz' to the old '.npz.tmp' filename,
# producing 'entity_scores.npz.tmp.npz' before the rename failed.
legacy_temporary = Path(str(SCORE_CACHE.with_suffix(".npz.tmp")) + ".npz")
if not SCORE_CACHE.is_file() and legacy_temporary.is_file():
    try:
        with np.load(legacy_temporary, allow_pickle=False) as archive:
            legacy_keys = archive["image_keys"].astype(str).tolist()
            legacy_entities = archive["entity_names"].astype(str).tolist()
            legacy_fingerprint = str(archive["fingerprint"].item())
            legacy_scores = archive["scores"]
        expected_shape = (len(legacy_keys), len(ENTITY_NAMES))
        if (legacy_entities == ENTITY_NAMES
                and legacy_fingerprint == SCORE_CACHE_FINGERPRINT
                and legacy_scores.shape == expected_shape):
            legacy_temporary.replace(SCORE_CACHE)
            print(f"Recovered interrupted score cache: {SCORE_CACHE}")
        else:
            print(f"Legacy temporary cache is incompatible; leaving it untouched: {legacy_temporary}")
    except Exception as exc:
        print(f"Could not recover legacy temporary cache {legacy_temporary}: {exc}")

cached_scores = {}
if SCORE_CACHE.is_file():
    with np.load(SCORE_CACHE, allow_pickle=False) as archive:
        keys = archive["image_keys"].tolist()
        matrix = archive["scores"]
        cached_entities = [str(name) for name in archive["entity_names"].tolist()]
        cached_fingerprint = (str(archive["fingerprint"].item())
                              if "fingerprint" in archive else None)
    if cached_entities != ENTITY_NAMES:
        print("Entity vocabulary changed since the cache was written; recomputing all scores.")
    elif cached_fingerprint != SCORE_CACHE_FINGERPRINT:
        print(f"Cache fingerprint differs (cached {cached_fingerprint}, now "
              f"{SCORE_CACHE_FINGERPRINT}): a prompt phrase, template, view, or model revision "
              "changed. Recomputing all scores rather than reusing stale ones.")
    else:
        cached_scores = {str(key): matrix[index] for index, key in enumerate(keys)}
        print(f"Cache hit: {len(cached_scores):,} images")


@torch.inference_mode()
def encode_images(image_paths):
    images = []
    for path in image_paths:
        with Image.open(path) as handle:
            images.append(handle.convert("RGB"))
    if backend == "open_clip":
        batch = torch.stack([preprocess(image) for image in images]).to(device)
        features = model.encode_image(batch)
    else:
        batch = preprocess(images=images, return_tensors="pt").to(device)
        features = model.get_image_features(**batch)
    for image in images:
        image.close()
    return torch.nn.functional.normalize(features.float(), dim=-1)


def score_entities(image_features):
    # Contrastive pair softmax. logit_scale is BiomedCLIP's learned temperature; using it keeps
    # the scores on the scale the model was trained to produce rather than an arbitrary one.
    scale = float(getattr(model, "logit_scale", torch.tensor(0.0)).exp()) if hasattr(
        model, "logit_scale") else 100.0
    positive_logits = image_features @ positive_matrix.T * scale
    negative_logits = image_features @ negative_matrix.T * scale
    stacked = torch.stack([positive_logits, negative_logits], dim=-1)
    return torch.softmax(stacked, dim=-1)[..., 0]      # P(positive phrase | image)


# ---- Encoder sanity preflight -------------------------------------------------------------
# NB 07's lesson in the form it takes here. BiomedCLIP will not "fail to receive" the image the
# way a chat template can, but it can return degenerate embeddings -- and constant embeddings
# would sail through to a full cohort of scores that all sit at 0.5, which looks like a weak
# agent rather than a broken encoder. Three different images must produce three different
# embeddings; anything else is a broken image path, not a weak model.
_probe_paths = [row[SCORE_VIEW] for _, row in cohort.head(3).iterrows()
                if isinstance(row.get(SCORE_VIEW), str)]
if len(_probe_paths) >= 2:
    _probe_features = encode_images(_probe_paths)
    _pairwise = (_probe_features @ _probe_features.T).cpu().numpy()
    _off_diagonal = [_pairwise[a][b] for a in range(len(_probe_paths))
                     for b in range(len(_probe_paths)) if a != b]
    print(f"Encoder preflight: {len(_probe_paths)} distinct images, "
          f"pairwise cosine {min(_off_diagonal):.4f}-{max(_off_diagonal):.4f}")
    if max(_off_diagonal) > 0.999:
        raise RuntimeError(
            "Encoder preflight FAILED: different radiographs produce near-identical embeddings "
            f"(max pairwise cosine {max(_off_diagonal):.5f}). The encoder is not responding to "
            "image content, so every entity score would collapse toward 0.5 and the construct-"
            "validity gate would blame the vocabulary. Check that SCORE_VIEW paths point at "
            "real, differing files and that preprocess() is the transform paired with this "
            "checkpoint."
        )
    print("  Embeddings vary across images, so the encoder is responding to content.")
else:
    print("Encoder preflight skipped: fewer than two probe images available.")

todo = [(str(row["image_key"]), row[SCORE_VIEW]) for _, row in cohort.iterrows()
        if str(row["image_key"]) not in cached_scores
        and isinstance(row[SCORE_VIEW], str) and row[SCORE_VIEW]]

def flush_scores(label):
    # Atomic write, so an interruption cannot leave a truncated cache that reads as valid.
    keys = sorted(cached_scores)
    if not keys:
        return
    temporary = SCORE_CACHE.with_suffix(".npz.tmp")
    # Passing a path ending in '.tmp' makes NumPy append another '.npz'.
    # An explicit binary handle preserves the exact temporary filename.
    with temporary.open("wb") as handle:
        np.savez_compressed(
            handle, image_keys=np.asarray(keys),
            entity_names=np.asarray(ENTITY_NAMES),
            fingerprint=np.asarray(SCORE_CACHE_FINGERPRINT),
            scores=np.stack([cached_scores[key] for key in keys]).astype(np.float32))
        handle.flush()
        os.fsync(handle.fileno())
    temporary.replace(SCORE_CACHE)
    print(f"  {label}: cache now holds {len(keys):,} images")


if todo:
    print(f"Encoding {len(todo):,} images")
    started = time.perf_counter()
    since_flush = 0
    for offset in range(0, len(todo), IMAGE_BATCH_SIZE):
        chunk = todo[offset:offset + IMAGE_BATCH_SIZE]
        try:
            features = encode_images([item[1] for item in chunk])
            scores = score_entities(features).cpu().numpy()
        except Exception:
            if since_flush:
                flush_scores("partial flush after error")
            raise
        for (key, _), vector in zip(chunk, scores):
            cached_scores[key] = vector
        since_flush += len(chunk)
        if since_flush >= SCORE_FLUSH_EVERY:
            flush_scores("checkpoint")
            since_flush = 0
        if (offset // IMAGE_BATCH_SIZE) % 10 == 0:
            done = min(offset + IMAGE_BATCH_SIZE, len(todo))
            rate = done / max(time.perf_counter() - started, 1e-6)
            print(f"    {done}/{len(todo)} ({rate:.1f} img/s)")
    flush_scores("final")

del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

score_frame = pd.DataFrame(
    [cached_scores[str(key)] for key in cohort["image_key"] if str(key) in cached_scores],
    columns=ENTITY_NAMES,
)
score_frame.insert(0, "image_key", [str(key) for key in cohort["image_key"]
                                    if str(key) in cached_scores])
try:
    score_frame.to_parquet(NB08_DIR / "entity_scores.parquet", index=False)
except Exception:
    score_frame.to_csv(NB08_DIR / "entity_scores.csv", index=False)
print(f"Entity score table: {score_frame.shape}")
print(score_frame[["image_key"] + ENTITY_NAMES[:5]].head().to_string(index=False))

## 7. Construct validity

There is **no entity ground truth** in this cohort, so the agent cannot be validated directly.
It can be validated *indirectly*, and this section is the difference between an agent that works
and a plausible-looking noise generator.

Four pre-registered checks:

1. **Opacity entities correlate with mRALE.** "extensive involvement", "dense opacity" and
   friends should rise with the annotated severity score.
2. **"Clear lungs" anti-correlates with mRALE.** A negative control; if this comes out positive,
   the sign convention or the prompt pairing is wrong.
3. **Montgomery normals score lower on opacity than MIDRC severe cases.** External face validity
   across a genuine domain gap.
4. **Montgomery TB shows a different entity signature than COVID** — cavitation and upper-lobe
   infiltrate should be relatively elevated. This is the differential-diagnosis evidence that
   E9b needs.

Check 5 is the confound audit: how well do **support devices alone** predict mRALE? If devices
rival opacity findings, the agent is partly reading the ICU context rather than the lungs, and
that has to be stated in the paper.

In [ ]:
from scipy.stats import spearmanr

labelled = internal_cohort.merge(score_frame, on="image_key", how="inner")
print(f"Internal images with scores and labels: {len(labelled):,}")

validity_rows = []
mrale = labelled["mrale_total_annotated"].astype(float).to_numpy()
for group, entity in ENTITY_INDEX:
    values = labelled[entity].astype(float).to_numpy()
    rho, p_value = spearmanr(values, mrale)
    covid_binary = (labelled["covid_positive"] == "Yes").astype(int).to_numpy()
    covid_rho, covid_p = spearmanr(values, covid_binary)
    validity_rows.append({
        "group": group, "entity": entity,
        "mean_score": round(float(values.mean()), 4),
        "std_score": round(float(values.std()), 4),
        "spearman_vs_mrale": round(float(rho), 4),
        "p_vs_mrale": float(p_value),
        "spearman_vs_covid": round(float(covid_rho), 4),
        "p_vs_covid": float(covid_p),
    })

validity = pd.DataFrame(validity_rows).sort_values("spearman_vs_mrale", ascending=False)
validity.to_csv(NB08_DIR / "construct_validity.csv", index=False)

pd.set_option("display.width", 200)
print()
print("Entity vs mRALE (Spearman), sorted:")
print(validity[["group", "entity", "mean_score", "spearman_vs_mrale",
                "spearman_vs_covid"]].to_string(index=False))

opacity_rho = validity[validity["entity"].isin(OPACITY_ENTITIES)]["spearman_vs_mrale"]
clear_rho = validity[validity["entity"] == "clear_lungs"]["spearman_vs_mrale"]
device_rho = validity[validity["entity"].isin(DEVICE_ENTITIES)]["spearman_vs_mrale"]

print()
print(f"CHECK 1  max opacity-entity rho vs mRALE : {opacity_rho.max():.4f} "
      f"(floor {MIN_OPACITY_MRALE_SPEARMAN})")
print(f"CHECK 2  clear_lungs rho vs mRALE        : "
      f"{float(clear_rho.iloc[0]) if len(clear_rho) else float('nan'):.4f} "
      f"(ceiling {MAX_CLEAR_LUNG_MRALE_SPEARMAN})")
print(f"CHECK 5  max support-device rho vs mRALE : {device_rho.max():.4f}")
if len(device_rho) and device_rho.max() >= opacity_rho.max():
    print("         ^ DEVICES PREDICT SEVERITY AS WELL AS OPACITY DOES. The agent is partly")
    print("           reading ICU context rather than lung parenchyma. This is a real confound")
    print("           and belongs in the limitations section, not in a footnote.")

In [ ]:
# Checks 3 and 4: cohort contrasts.
cohort_rows = []
if len(external_cohort):
    external_scored = external_cohort.merge(score_frame, on="image_key", how="inner")
    severe = labelled[labelled["severity_band"] == "severe"]
    none_band = labelled[labelled["severity_band"] == "none"]
    groups = {
        "MIDRC_severe(19-24)": severe,
        "MIDRC_none(0)": none_band,
    }
    for subcohort in sorted(external_scored["subcohort"].dropna().unique()):
        groups[str(subcohort)] = external_scored[external_scored["subcohort"] == subcohort]

    for entity in ENTITY_NAMES:
        row = {"entity": entity}
        for name, frame in groups.items():
            row[name] = round(float(frame[entity].astype(float).mean()), 4) if len(frame) else None
        cohort_rows.append(row)

    contrast = pd.DataFrame(cohort_rows)
    contrast.to_csv(NB08_DIR / "cohort_entity_contrast.csv", index=False)
    print("Mean entity score by cohort:")
    print(contrast.to_string(index=False))

    if "X1a_normal" in groups and len(groups["X1a_normal"]) and len(severe):
        opacity_severe = float(severe[OPACITY_ENTITIES].astype(float).mean().mean())
        opacity_normal = float(groups["X1a_normal"][OPACITY_ENTITIES].astype(float).mean().mean())
        print()
        print(f"CHECK 3  mean opacity score, MIDRC severe : {opacity_severe:.4f}")
        print(f"         mean opacity score, Montgomery normal: {opacity_normal:.4f}")
        print(f"         separation: {opacity_severe - opacity_normal:+.4f} "
              + ("(expected direction)" if opacity_severe > opacity_normal
                 else "(WRONG DIRECTION -- investigate before using this agent)"))

    if "X1b_tb" in groups and len(groups["X1b_tb"]):
        tb = groups["X1b_tb"]
        print()
        print("CHECK 4  TB signature vs MIDRC COVID (mean score difference):")
        for entity in ["cavitation", "upper_lobe_infiltrate", "ground_glass_opacity",
                       "bilateral_involvement"]:
            if entity in tb.columns:
                difference = float(tb[entity].astype(float).mean()) - float(
                    labelled[entity].astype(float).mean())
                print(f"           {entity:<24} {difference:+.4f}")
        print("         Cavitation and upper-lobe infiltrate should be relatively elevated in")
        print("         TB. This is the differential evidence E9b needs: it shows the framework")
        print("         can express 'abnormal but not COVID-like' rather than only a severity.")
else:
    contrast = pd.DataFrame()
    print("No external cohort scored; checks 3 and 4 skipped.")

## 8. Entity-feature probe

A **linear** probe over the entity scores, fitted per fold on inner-training data only. Linear
by choice: the coefficients are the interpretability deliverable, and a non-linear head would
trade that away for a small accuracy gain on a 30-dimensional feature vector.

The probe gives the agent a row in the shared prediction schema, so NB 13's registry consumes
it like any other agent and `E1-L6` can leave it out cleanly.

In [ ]:
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler

COMPONENT_COLUMNS = OrderedDict([
    ("extent_right", ("extent_right_numerical", 0, 4)),
    ("density_right", ("density_right_numerical", 0, 3)),
    ("extent_left", ("extent_left_numerical", 0, 4)),
    ("density_left", ("density_left_numerical", 0, 3)),
])

predictions = []
per_fold_metrics = {}
coefficient_rows = []
fold_models = {}

for fold in range(N_FOLDS):
    test_subset = labelled[labelled["fold"] == fold]
    pool = labelled[labelled["fold"] != fold]
    train_subset, validation_subset = grouped_inner_split(
        pool, INTERNAL_VALIDATION_FRACTION, SEED + fold)

    train_features = train_subset[ENTITY_NAMES].astype(float).to_numpy()
    test_features = test_subset[ENTITY_NAMES].astype(float).to_numpy()
    scaler = StandardScaler().fit(train_features) if STANDARDISE_FEATURES else None
    if scaler is not None:
        train_features = scaler.transform(train_features)
        test_features = scaler.transform(test_features)

    covid_train = (train_subset["covid_positive"] == "Yes").astype(int).to_numpy()
    classifier = None
    if len(set(covid_train.tolist())) == 2:
        classifier = LogisticRegression(
            C=PROBE_C, max_iter=2000, class_weight=None, random_state=SEED)
        classifier.fit(train_features, covid_train)
        covid_score = classifier.predict_proba(test_features)[:, 1]
    else:
        covid_score = np.full(len(test_subset), float(covid_train.mean()))

    regressors, component_predictions = {}, {}
    for component, (column, low, high) in COMPONENT_COLUMNS.items():
        targets = train_subset[column].astype(float).to_numpy()
        valid = ~np.isnan(targets)
        regressor = Ridge(alpha=PROBE_ALPHA, random_state=SEED)
        regressor.fit(train_features[valid], targets[valid])
        regressors[component] = regressor
        raw = regressor.predict(test_features)
        component_predictions[component] = np.clip(np.rint(raw), low, high).astype(int)
        for index, entity in enumerate(ENTITY_NAMES):
            coefficient_rows.append({
                "fold": fold, "target": component, "entity": entity,
                "coefficient": float(regressor.coef_[index]),
            })
    if classifier is not None:
        for index, entity in enumerate(ENTITY_NAMES):
            coefficient_rows.append({
                "fold": fold, "target": "covid", "entity": entity,
                "coefficient": float(classifier.coef_[0][index]),
            })

    fold_models[fold] = {"scaler": scaler, "classifier": classifier, "regressors": regressors}

    fold_rows = []
    for position, (_, source) in enumerate(test_subset.iterrows()):
        right = int(component_predictions["extent_right"][position]
                    * component_predictions["density_right"][position])
        left = int(component_predictions["extent_left"][position]
                   * component_predictions["density_left"][position])
        score = float(covid_score[position])
        entity_vector = {entity: round(float(source[entity]), 4) for entity in ENTITY_NAMES}
        fold_rows.append(cm.make_prediction_row(
            image_key=str(source["image_key"]), cohort=source.get("cohort", "MIDRC"),
            subcohort=source.get("subcohort", "MIDRC"), filename=source["filename"],
            held_out_fold=fold, agent=AGENT_NAME, arm=ENTITY_ARM, view="v0", task="joint",
            covid_pred="Yes" if score >= 0.5 else "No", covid_score=score,
            mrale_total=right + left, mrale_right=right, mrale_left=left,
            extent_right=int(component_predictions["extent_right"][position]),
            density_right=int(component_predictions["density_right"][position]),
            extent_left=int(component_predictions["extent_left"][position]),
            density_left=int(component_predictions["density_left"][position]),
            valid=True, parse_error=None,
            model_id=BIOMEDCLIP_ID, model_revision=BIOMEDCLIP_REVISION,
            entity_scores=entity_vector,
            **ground_truth_fields(source),
        ))
    predictions.extend(fold_rows)
    per_fold_metrics[fold] = evaluate_arm(fold_rows, f"{ENTITY_ARM}/fold{fold}")
    print(f"fold {fold}: train={len(train_subset)} test={len(test_subset)}")
    print_arm_summary(per_fold_metrics[fold])

cm.write_jsonl(NB08_DIR / "predictions_entity_probe.jsonl", predictions)
print()
print("POOLED out-of-fold:")
pooled = evaluate_arm(predictions, ENTITY_ARM)
print_arm_summary(pooled)

In [ ]:
coefficients = pd.DataFrame(coefficient_rows)
mean_coefficients = (coefficients.groupby(["target", "entity"])["coefficient"]
                     .agg(["mean", "std"]).reset_index()
                     .rename(columns={"mean": "coefficient_mean", "std": "coefficient_std"}))
mean_coefficients["stable_sign"] = (
    mean_coefficients["coefficient_mean"].abs() > mean_coefficients["coefficient_std"])
mean_coefficients.to_csv(NB08_DIR / "probe_coefficients.csv", index=False)

print("Which named findings drive the severity prediction (mean over folds)?")
print("A coefficient is reported as stable only when |mean| exceeds its across-fold SD.")
for target in ["covid", "extent_right", "density_right"]:
    subset = mean_coefficients[mean_coefficients["target"] == target].copy()
    if not len(subset):
        continue
    subset = subset.reindex(subset["coefficient_mean"].abs().sort_values(
        ascending=False).index).head(8)
    print()
    print(f"  --- {target}")
    for _, row in subset.iterrows():
        marker = " " if row["stable_sign"] else "  (unstable)"
        print(f"    {row['entity']:<26} {row['coefficient_mean']:+.4f} "
              f"+/- {row['coefficient_std']:.4f}{marker}")
print()
print("This table is a deliverable for referee 1.4: it states, in clinical vocabulary, what the")
print("agent is keying on. No other agent in the roster can produce it.")

### 8b. Device-confound ablation — is this agent reading lungs or the ICU?

The `support_device` entities were scored deliberately, because ICU hardware is a severity proxy
that requires no reading of the lung parenchyma. The construct-validity table shows the concern is
not hypothetical: `ecg_leads` correlates ρ = 0.673 with mRALE and `endotracheal_tube` ρ = 0.672,
against ρ = 0.775 for the best genuine opacity entity. In the `extent_right` probe, `ecg_leads`
is the **second strongest positive coefficient**, ahead of `dense_opacity`.

Correlation alone cannot separate "the agent exploits devices" from "sick patients have both
devices and opacities". This cell refits the probe three ways and lets the numbers decide:

| feature set | what it tests |
| --- | --- |
| all entities | the reported model |
| devices removed | what the agent achieves on lung findings alone |
| devices only | how much severity is predictable from care setting with no lung reading at all |

Two different conclusions follow depending on the outcome, and they matter for how the paper is
written. If removing devices barely costs anything, the agent's signal is genuinely parenchymal
and the device-free model is the clean number to headline. If devices alone rival the full model,
the benchmark itself is partly measuring care setting — a limitation of the task, not of this
agent, and one that applies to every arm.


In [ ]:
DEVICE_ENTITIES = list(ENTITY_VOCABULARY["support_device"])
LUNG_ENTITIES = [entity for entity in ENTITY_NAMES if entity not in DEVICE_ENTITIES]


def refit_probe(feature_names, label):
    """Out-of-fold mRALE from a Ridge probe on the given entity subset."""
    predicted, reference = [], []
    for fold in range(N_FOLDS):
        train = labelled[labelled["fold"] != fold]
        test = labelled[labelled["fold"] == fold]
        scaler = StandardScaler().fit(train[feature_names].to_numpy())
        train_x = scaler.transform(train[feature_names].to_numpy())
        test_x = scaler.transform(test[feature_names].to_numpy())
        values = {}
        for component, (column, low, high) in COMPONENT_COLUMNS.items():
            targets = train[column].astype(float).to_numpy()
            valid = ~np.isnan(targets)
            model = Ridge(alpha=PROBE_ALPHA, random_state=SEED)
            model.fit(train_x[valid], targets[valid])
            values[component] = np.clip(np.rint(model.predict(test_x)), low, high)
        total = (values["extent_right"] * values["density_right"]
                 + values["extent_left"] * values["density_left"])
        predicted += list(total)
        reference += list(test["mrale_total_annotated"].astype(float).to_numpy())
    rows = [{"gt_mrale_total": int(r), "mrale_total": int(p)}
            for r, p in zip(reference, predicted) if not math.isnan(r)]
    metrics = cm.mrale_metrics(rows)
    return {"feature_set": label, "n_features": len(feature_names),
            "mae": round(metrics["mae"], 3), "rmse": round(metrics["rmse"], 3),
            "qwk": round(metrics.get("qwk", float("nan")), 4),
            "spearman": round(metrics.get("spearman_rho", float("nan")), 4)}


device_rows = [
    refit_probe(ENTITY_NAMES, "all_entities"),
    refit_probe(LUNG_ENTITIES, "devices_removed"),
    refit_probe(DEVICE_ENTITIES, "devices_only"),
]
device_frame = pd.DataFrame(device_rows)
device_frame.to_csv(NB08_DIR / "device_confound_ablation.csv", index=False)
print(device_frame.to_string(index=False))

full = device_rows[0]["mae"]
nodev = device_rows[1]["mae"]
devonly = device_rows[2]["mae"]
device_cost = round(nodev - full, 3)
device_alone_gap = round(devonly - full, 3)
print()
print(f"  cost of removing devices : {device_cost:+.3f} MAE")
print(f"  devices alone vs full    : {device_alone_gap:+.3f} MAE")
print()
if abs(device_cost) <= 0.25:
    print("  The agent's signal is genuinely PARENCHYMAL: dropping every device feature costs")
    print(f"  {device_cost:+.3f} MAE. Headline the device-free model -- it removes the confound")
    print("  outright at negligible cost, which is the cleanest position to defend.")
    RECOMMENDED_FEATURE_SET = "devices_removed"
else:
    print(f"  Removing devices costs {device_cost:+.3f} MAE, so part of the reported accuracy")
    print("  depends on care-setting cues. Report both numbers and treat the device-free one as")
    print("  the primary result.")
    RECOMMENDED_FEATURE_SET = "report_both"

print()
print("  SEPARATE POINT, about the task rather than this agent:")
print(f"  four ICU-hardware detectors ALONE reach MAE {devonly:.3f} "
      f"(spearman {device_rows[2]['spearman']:.3f}).")
print("  Compare the zero-shot VLM arms in NB 07. If device presence alone beats a 4B VLM at")
print("  mRALE, then severity benchmarks drawn from ICU cohorts partly measure care setting,")
print("  and that caveat applies to EVERY arm in Table 2 -- including the framework. It belongs")
print("  in the limitations section regardless of what this agent does.")

cm.write_json(NB08_DIR / "device_confound.json", {
    "device_entities": DEVICE_ENTITIES,
    "results": device_rows,
    "device_removal_cost_mae": device_cost,
    "devices_alone_gap_mae": device_alone_gap,
    "recommended_feature_set": RECOMMENDED_FEATURE_SET,
    "interpretation": (
        "Removing devices costs little => the agent reads lung findings, not ICU context. "
        "Devices alone still predicting severity reasonably well is a property of the COHORT "
        "(sicker patients receive more hardware) and caveats every arm, not just this one."
    ),
})


## 9. Structured findings for the reasoner

What the E3 reasoner actually consumes. Scores are thresholded, ranked, capped, and paired with
a confidence, so the prompt stays short and every claim in a reasoning trace can be traced back
to a specific agent output during the E10c grounding audit.

In [ ]:
def structured_findings(image_key, scores):
    present = [
        {"finding": entity, "confidence": round(float(scores[entity]), 3),
         "group": group}
        for group, entity in ENTITY_INDEX
        if float(scores[entity]) >= FINDING_THRESHOLD
    ]
    present.sort(key=lambda item: item["confidence"], reverse=True)
    truncated = present[:FINDING_MAX_REPORTED]
    # Entropy over the pair softmax, averaged: high means the agent is unsure across the board.
    probabilities = np.asarray([float(scores[entity]) for entity in ENTITY_NAMES])
    probabilities = np.clip(probabilities, 1e-6, 1 - 1e-6)
    entropy = float(np.mean(-(probabilities * np.log(probabilities)
                              + (1 - probabilities) * np.log(1 - probabilities))))
    return {
        "image_key": image_key,
        "agent": AGENT_NAME,
        "findings": truncated,
        "n_findings_above_threshold": len(present),
        "n_findings_reported": len(truncated),
        "mean_binary_entropy": round(entropy, 4),
        "threshold": FINDING_THRESHOLD,
    }


findings_rows = [structured_findings(str(row["image_key"]), row)
                 for _, row in score_frame.iterrows()]
cm.write_jsonl(NB08_DIR / "entity_findings.jsonl", findings_rows)

counts = Counter(finding["finding"] for row in findings_rows for finding in row["findings"])
print(f"Structured findings written for {len(findings_rows):,} images "
      f"(threshold {FINDING_THRESHOLD})")
print(f"Mean findings reported per image: "
      f"{np.mean([row['n_findings_reported'] for row in findings_rows]):.2f}")
print()
print("Most frequently reported findings:")
for entity, count in counts.most_common(12):
    print(f"  {entity:<26} {count:>6} ({count / max(len(findings_rows), 1):.1%})")

empty = sum(1 for row in findings_rows if row["n_findings_reported"] == 0)
saturated = sum(1 for row in findings_rows
                if row["n_findings_above_threshold"] >= len(ENTITY_NAMES) * 0.8)
print()
print(f"Images with NO finding above threshold : {empty} ({empty / max(len(findings_rows), 1):.1%})")
print(f"Images with >=80% of entities firing   : {saturated} "
      f"({saturated / max(len(findings_rows), 1):.1%})")
if saturated > 0.2 * len(findings_rows):
    print("  A large saturated fraction means the contrastive pairs are biased toward the")
    print("  positive phrase rather than discriminating. Raise FINDING_THRESHOLD or rewrite")
    print("  the negative prompts before feeding this to the reasoner.")

print()
print("Example payload as the reasoner will see it:")
print(json.dumps(findings_rows[0], indent=2)[:700])

## 10. External cohorts

In [ ]:
external_predictions = []
if RUN_EXTERNAL and len(external_cohort):
    external_scored = external_cohort.merge(score_frame, on="image_key", how="inner")
    if len(external_scored):
        features = external_scored[ENTITY_NAMES].astype(float).to_numpy()
        accumulated_score = np.zeros(len(external_scored))
        accumulated_components = {component: np.zeros(len(external_scored))
                                  for component in COMPONENT_COLUMNS}
        n_models = 0
        for fold, payload in fold_models.items():
            transformed = (payload["scaler"].transform(features)
                           if payload["scaler"] is not None else features)
            if payload["classifier"] is not None:
                accumulated_score += payload["classifier"].predict_proba(transformed)[:, 1]
            for component, regressor in payload["regressors"].items():
                accumulated_components[component] += regressor.predict(transformed)
            n_models += 1
        accumulated_score /= max(n_models, 1)
        for component in accumulated_components:
            accumulated_components[component] /= max(n_models, 1)

        for position, (_, source) in enumerate(external_scored.iterrows()):
            bounds = {c: (low, high) for c, (_, low, high) in COMPONENT_COLUMNS.items()}
            values = {
                component: int(np.clip(round(accumulated_components[component][position]),
                                       *bounds[component]))
                for component in COMPONENT_COLUMNS
            }
            right = values["extent_right"] * values["density_right"]
            left = values["extent_left"] * values["density_left"]
            score = float(accumulated_score[position])
            external_predictions.append(cm.make_prediction_row(
                image_key=str(source["image_key"]), cohort=source.get("cohort"),
                subcohort=source.get("subcohort"), filename=source["filename"],
                held_out_fold=None, agent=AGENT_NAME, arm=ENTITY_ARM, view="v0", task="joint",
                covid_pred="Yes" if score >= 0.5 else "No", covid_score=score,
                mrale_total=right + left, mrale_right=right, mrale_left=left,
                extent_right=values["extent_right"], density_right=values["density_right"],
                extent_left=values["extent_left"], density_left=values["density_left"],
                valid=True, parse_error=None,
                model_id=BIOMEDCLIP_ID, model_revision=BIOMEDCLIP_REVISION,
                ensemble_of_folds=n_models,
                **ground_truth_fields(source),
            ))
        cm.write_jsonl(NB08_DIR / "external_predictions.jsonl", external_predictions)

        external_summary = []
        for subcohort in sorted({row["subcohort"] for row in external_predictions}):
            rows = [row for row in external_predictions if row["subcohort"] == subcohort]
            covid = evaluate_arm(rows, ENTITY_ARM).get("covid", {})
            external_summary.append({
                "arm": ENTITY_ARM, "subcohort": subcohort, "n": len(rows),
                "specificity": round(covid.get("specificity", float("nan")), 4),
                "mean_predicted_mrale": round(
                    float(np.mean([row["mrale_total"] for row in rows])), 2),
            })
        frame = pd.DataFrame(external_summary)
        frame.to_csv(NB08_DIR / "external_summary.csv", index=False)
        print(frame.to_string(index=False))
else:
    print("External prediction skipped.")

## 11. Arm summary, run configuration, and gate

In [ ]:
aggregate = cm.aggregate_over_folds(per_fold_metrics)
pd.DataFrame(aggregate).to_csv(
    NB08_DIR / f"cross_validation_aggregate_95ci_{ENTITY_ARM}.csv", index=False)


def fold_ci(metric):
    match = [row for row in aggregate if row["metric"] == metric]
    return ((match[0]["mean"], match[0]["ci95_lower"], match[0]["ci95_upper"])
            if match else (None, None, None))


covid, mrale = pooled.get("covid", {}), pooled.get("mrale", {})
mae_mean, mae_low, mae_high = fold_ci("mrale.mae")
auroc_mean, auroc_low, auroc_high = fold_ci("covid.auroc")

summary = pd.DataFrame([OrderedDict([
    ("arm", ENTITY_ARM), ("agent", AGENT_NAME), ("model_id", BIOMEDCLIP_ID),
    ("adaptation", "frozen + linear probe"),
    ("n_features", len(ENTITY_NAMES)), ("n_images", pooled["n_rows"]),
    ("mrale_mae_pooled", round(mrale.get("mae", float("nan")), 3)),
    ("mrale_mae_ci95", None if mae_mean is None else f"[{mae_low:.3f}, {mae_high:.3f}]"),
    ("covid_auroc_pooled", round(covid.get("auroc", float("nan")), 4)),
    ("covid_auroc_ci95", None if auroc_mean is None else f"[{auroc_low:.4f}, {auroc_high:.4f}]"),
    ("covid_auprc", round(covid.get("auprc", float("nan")), 4)),
    ("covid_balanced_accuracy", round(covid.get("balanced_accuracy", float("nan")), 4)),
    ("covid_sensitivity", round(covid.get("sensitivity", float("nan")), 4)),
    ("covid_specificity", round(covid.get("specificity", float("nan")), 4)),
    ("covid_f1", round(covid.get("f1", float("nan")), 4)),
    ("covid_mcc", round(covid.get("mcc", float("nan")), 4)),
    ("covid_brier", round(covid.get("brier", float("nan")), 4)),
    ("covid_ece", round(covid.get("ece", float("nan")), 4)),
    ("mrale_rmse", round(mrale.get("rmse", float("nan")), 3)),
    ("mrale_qwk", round(mrale.get("qwk", float("nan")), 4)),
    ("mrale_spearman", round(mrale.get("spearman_rho", float("nan")), 4)),
    ("mrale_within1", round(mrale.get("within1_accuracy", float("nan")), 4)),
    ("mae_band_none", round(mrale.get("mae_band_none", float("nan")), 3)),
    ("mae_band_mild", round(mrale.get("mae_band_mild", float("nan")), 3)),
    ("mae_band_moderate", round(mrale.get("mae_band_moderate", float("nan")), 3)),
    ("mae_band_severe", round(mrale.get("mae_band_severe", float("nan")), 3)),
])])
summary.to_csv(NB08_DIR / "arm_summary.csv", index=False)
cm.write_json(NB08_DIR / "per_fold_metrics.json", per_fold_metrics)
print(summary.to_string(index=False))

In [ ]:
cm.write_json(NB08_DIR / "run_config.json", {
    "written_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "08_biomedclip_entity_agent.ipynb",
    "protocol_agent": "A6_biomedclip (reinstated; see notebook header)",
    "protocol_experiments": [ENTITY_ARM, "E1-L6"],
    "referee_point": "1.2 -- named component whose contribution must be disentangled",
    "seed": SEED,
    "model": {"model_id": BIOMEDCLIP_ID, "revision": BIOMEDCLIP_REVISION,
              "revision_source": BIOMEDCLIP_REVISION_SOURCE,
              "backend": backend, "parameters": int(n_parameters), "frozen": True},
    "vocabulary": {"n_entities": len(ENTITY_NAMES),
                   "groups": {g: list(e) for g, e in ENTITY_VOCABULARY.items()},
                   "prompt_templates": PROMPT_TEMPLATES,
                   "scoring": "contrastive positive/negative pair softmax at the model's "
                              "learned logit scale"},
    "findings": {"threshold": FINDING_THRESHOLD, "max_reported": FINDING_MAX_REPORTED},
    "probe": {"type": "linear (LogisticRegression + Ridge)", "C": PROBE_C,
              "alpha": PROBE_ALPHA, "standardised": STANDARDISE_FEATURES,
              "rationale": "Linear by choice: the coefficients are the interpretability "
                           "deliverable for referee 1.4."},
    "construct_validity_floors": {
        "min_opacity_mrale_spearman": MIN_OPACITY_MRALE_SPEARMAN,
        "max_clear_lung_mrale_spearman": MAX_CLEAR_LUNG_MRALE_SPEARMAN,
    },
    "known_confound": "support_device entities are scored explicitly because ICU device "
                      "presence is a severity proxy that does not require reading the lungs.",
    "max_images": MAX_IMAGES,
})

# Usability flags use NB 07's semantics so NB 13 and later consumers can normalize them.
# A6 is judged on its E1-L6 leave-one-out delta rather than this standalone row, but the row
# still must not be liftable into Table 2 without its health flags attached.
usability = {ENTITY_ARM: {"entity_scores_usable": None, "probe_usable": None}}

failures, warnings = [], []

expected = {str(key) for key in internal_cohort["image_key"]}
scored = set(score_frame["image_key"])
missing = expected - scored
if missing:
    failures.append(f"{len(missing)} internal images have no entity scores "
                    f"(e.g. {sorted(missing)[:3]}).")

covered = {row["image_key"] for row in predictions}
if covered != expected:
    failures.append(f"Probe out-of-fold coverage {len(covered)} of {len(expected)}.")
repeated = [key for key, count in Counter(
    row["image_key"] for row in predictions).items() if count > 1]
if repeated:
    failures.append(f"{len(repeated)} images predicted more than once.")

if degenerate:
    failures.append(
        f"{len(degenerate)} entity prompt pairs are near-degenerate "
        f"({[name for name, _ in degenerate][:3]}): the positive and negative phrases embed "
        "almost identically, so those entity scores carry no information. Rewrite the "
        "negative phrases."
    )

# Construct validity is the only evidence this agent measures what its labels claim, so a
# failure here BLOCKS by design: feeding a noise channel to the E3 reasoner is worse than
# omitting the agent. If it cannot be made to pass, the honest resolution is to drop A6 --
# set ACKNOWLEDGE_A6_UNUSABLE = True, which records the agent as unusable, omits it from
# Table 2 and the E1 roster, and answers referee 1.2 with "no validated entity signal could be
# obtained" rather than with a fabricated contribution.
ACKNOWLEDGE_A6_UNUSABLE = False

max_opacity = float(opacity_rho.max()) if len(opacity_rho) else float("nan")
if math.isnan(max_opacity) or max_opacity < MIN_OPACITY_MRALE_SPEARMAN:
    failures.append(
        f"CONSTRUCT VALIDITY: best opacity-entity correlation with mRALE is "
        f"{max_opacity:.4f}, below the pre-registered floor of "
        f"{MIN_OPACITY_MRALE_SPEARMAN}. With no entity ground truth, this check is the only "
        "evidence the agent measures what its labels claim. Do not feed it to the reasoner "
        "until this passes."
    )

if len(clear_rho):
    clear_value = float(clear_rho.iloc[0])
    if clear_value > MAX_CLEAR_LUNG_MRALE_SPEARMAN:
        failures.append(
            f"CONSTRUCT VALIDITY: 'clear_lungs' correlates {clear_value:+.4f} with mRALE, "
            f"above the ceiling of {MAX_CLEAR_LUNG_MRALE_SPEARMAN}. A clear-lung score that "
            "rises with severity means the sign convention or the prompt pairing is inverted."
        )

max_device = float(device_rho.max()) if len(device_rho) else float("nan")
if not math.isnan(max_device) and not math.isnan(max_opacity) and max_device >= max_opacity:
    warnings.append(
        f"Support-device entities predict mRALE at least as well as opacity entities "
        f"({max_device:.4f} vs {max_opacity:.4f}). The agent is partly reading ICU context. "
        "Disclose this in the limitations and consider reporting device-adjusted results."
    )

if saturated > 0.2 * len(findings_rows):
    warnings.append(f"{saturated / max(len(findings_rows), 1):.1%} of images fire >=80% of "
                    "entities. The contrastive pairs are biased positive; raise the threshold "
                    "or rewrite the negative prompts before the E3 reasoner consumes them.")

if MAX_IMAGES is not None:
    warnings.append(f"WIRING CHECK MODE: only {MAX_IMAGES} images. Set MAX_IMAGES=None.")

# The COVID head is uninformative here (entity-vs-PCR correlations are all ~0.0-0.08), so its
# probe coefficients fit noise and must NOT be presented as interpretability. Checked rather
# than assumed, because a coefficient table looks authoritative whether or not it means anything.
covid_mcc = pooled.get("covid", {}).get("mcc", float("nan"))
covid_balanced = pooled.get("covid", {}).get("balanced_accuracy", float("nan"))
if not math.isnan(covid_mcc) and abs(covid_mcc) < 0.05:
    warnings.append(
        f"COVID head is uninformative (MCC {covid_mcc:+.4f}, balanced accuracy "
        f"{covid_balanced:.4f}): it predicts one class for essentially every image. Report the "
        "mRALE side of this agent only. Do NOT present the 'covid' rows of "
        "probe_coefficients.csv as interpretability -- with MCC at zero they are fitted noise, "
        "and they read as clinically incoherent (clear_lungs positively predicting COVID). The "
        "high AUPRC is prevalence, not skill."
    )

if "device_cost" in dir() and abs(device_cost) > 0.25:
    warnings.append(
        f"Device-confound ablation: removing the {len(DEVICE_ENTITIES)} support-device entities "
        f"costs {device_cost:+.3f} MAE, so part of the reported accuracy rests on care-setting "
        "cues rather than lung parenchyma. Report the device-free model as primary."
    )

construct_validity_failed = any("CONSTRUCT VALIDITY" in message for message in failures)
usability[ENTITY_ARM].update({
    "entity_scores_usable": not construct_validity_failed,
    "probe_usable": not construct_validity_failed,
    "max_opacity_mrale_spearman": (None if math.isnan(max_opacity)
                                   else round(max_opacity, 4)),
    "max_device_mrale_spearman": (None if math.isnan(max_device)
                                  else round(max_device, 4)),
})

if construct_validity_failed and ACKNOWLEDGE_A6_UNUSABLE:
    failures = [message for message in failures if "CONSTRUCT VALIDITY" not in message]
    warnings.append(
        "A6 DROPPED by operator acknowledgement: construct validity failed and the agent is "
        "recorded as unusable. Omit it from Table 2 and the E1 roster, drop E1-L6, and state "
        "in the manuscript that no validated entity signal could be obtained. That is a "
        "defensible answer to referee 1.2; a fabricated contribution is not."
    )

cm.write_json(NB08_DIR / "usability.json", usability)

summary_path = NB08_DIR / "arm_summary.csv"
if summary_path.is_file():
    stamped = pd.read_csv(summary_path)
    flags = usability[ENTITY_ARM]
    stamped["entity_scores_usable"] = flags["entity_scores_usable"]
    stamped["probe_usable"] = flags["probe_usable"]
    stamped["report_in_table2"] = ("full row" if flags["probe_usable"]
                                   else "OMIT -- construct validity failed")
    stamped.to_csv(summary_path, index=False)
    print("arm_summary.csv stamped:",
          {key: stamped.iloc[0][key] for key in
           ["arm", "entity_scores_usable", "probe_usable", "report_in_table2"]})

warnings.append(
    f"E1-L6 INPUT: agent A6 standalone reaches mRALE MAE "
    f"{mrale.get('mae', float('nan')):.3f} and AUROC {covid.get('auroc', float('nan')):.4f}. "
    "Its value in the framework is the named-finding evidence channel, not standalone "
    "accuracy -- judge it on the E1-L6 leave-one-out delta, not on this row."
)


def report(title, messages):
    print(title)
    if messages:
        for message in messages:
            print("  -", message)
    else:
        print("  none")


report("WARNINGS", warnings)
print()
report("FAILURES", failures)

cm.write_json(NB08_DIR / "gate_nb08.json",
              {"passed": not failures, "failures": failures, "warnings": warnings,
               "max_opacity_mrale_spearman": max_opacity,
               "max_device_mrale_spearman": max_device})
# Reasons go in the message so a pasted traceback is self-explanatory (NB 07 lesson).
if failures:
    detail = "\n".join(f"  [{index + 1}] {message}"
                       for index, message in enumerate(failures))
    raise AssertionError(
        f"NB 08 gate failed with {len(failures)} blocking issue(s):\n{detail}")
print()
print("NB 08 gate: PASSED")

## Notes carried forward

- **`entity_findings.jsonl` is the reasoner's evidence channel.** NB 14's E3 prompts read it,
  and NB 20's E10c grounding audit checks reasoning traces against it — an unsupported claim is
  one that cites a finding this agent never reported.
- **`probe_coefficients.csv` is an interpretability deliverable in its own right.** It states in
  clinical vocabulary what the agent keys on, per fold, with stability marked. No other agent in
  the roster produces anything comparable, and it costs ~20 minutes of GPU time.
- **Judge A6 on E1-L6, not on its standalone row.** Its accuracy will likely trail the LoRA
  VLMs. That is fine — its job is to supply named evidence, and the leave-one-out delta is what
  measures whether that evidence helps.
- **The device confound is now measurable.** If `support_device` entities track mRALE as well as
  opacity does, the framework is partly reading ICU context. Better to surface that ourselves
  than to have a referee ask.
- **Plan numbering shifts from here**: MedGemma LoRA → NB 09, Qwen LoRA → NB 10, NV-Reason →
  NB 11, anatomy-aware → NB 12, agent registry → NB 13, fusion → NB 14, reasoner → NB 15.
- `open_clip_torch` must be installed. The loader tries transformers as a fallback but will
  raise rather than substitute another model, because E1-L6 has to be about BiomedCLIP
  specifically for the referee's question to be answered.